In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. LOAD DATA 
df = pd.read_csv('DATASET_TA1(Sheet1).csv', sep=';', decimal=',')

# 2. DATA PREPARATION (Cleaning, Imputasi & Outlier)
selected_features = ['Hemoglobin', 'Hematokrit', 'HbA1c', 'RBG']
data_cluster = df[selected_features].copy()

# A. Imputasi Missing Value 
data_cluster = data_cluster.fillna(data_cluster.median())

# B. Deteksi & Penghapusan Outlier dengan IQR (Interquartile Range)
kondisi_normal = pd.Series(True, index=data_cluster.index)

for col in selected_features:
    Q1 = data_cluster[col].quantile(0.25)
    Q3 = data_cluster[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Memperbarui kondisi_normal: Baris harus berada di dalam batas IQR
    kondisi_normal = kondisi_normal & (data_cluster[col] >= lower_bound) & (data_cluster[col] <= upper_bound)

# Menerapkan filter kondisi_normal ke data clustering dan dataframe asli
data_cluster = data_cluster[kondisi_normal]
df_bersih = df[kondisi_normal].copy() # Menggunakan df_bersih agar tidak merusak df asli

print(f"Jumlah data awal: {len(df)} baris")
print(f"Jumlah data setelah pembersihan outlier: {len(df_bersih)} baris\n")

# C. Standarisasi Data (Z-Score)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_cluster)

# 3. K-MEANS CLUSTERING (k = 2)
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
df_bersih['Cluster'] = kmeans.fit_predict(data_scaled)

# 4. EKSTRAKSI HASIL (CENTROID)
centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)
df_centroid = pd.DataFrame(centroids_original, columns=selected_features)
df_centroid.index = ['Cluster 0', 'Cluster 1']

print("=== DISTRIBUSI ANGGOTA CLUSTER (DATA BERSIH) ===")
print(df_bersih['Cluster'].value_counts())

print("\n=== PUSAT CLUSTER (CENTROID) SETELAH OUTLIER DIHAPUS ===")
print(df_centroid)

print("\n=== TABEL SILANG CLUSTER VS DIAGNOSE ===")
cross_tab = pd.crosstab(df_bersih['Cluster'], df_bersih['Diagnose'])
print(cross_tab)

Jumlah data awal: 1641 baris
Jumlah data setelah pembersihan outlier: 1440 baris

=== DISTRIBUSI ANGGOTA CLUSTER (DATA BERSIH) ===
1    842
0    598
Name: Cluster, dtype: int64

=== PUSAT CLUSTER (CENTROID) SETELAH OUTLIER DIHAPUS ===
           Hemoglobin  Hematokrit      HbA1c         RBG
Cluster 0   10.747739   31.921156  10.836080  290.818258
Cluster 1   14.180308   41.738790  11.631756  380.683274

=== TABEL SILANG CLUSTER VS DIAGNOSE ===
Diagnose  DM TYPE 2  DM TYPE 2 + Penyerta
Cluster                                  
0               343                   255
1               519                   323
